# Capstone: Which Labels Should You Audit?

Training sets have wrong labels in them. Not a few adversarial ones, just the ordinary
result of annotation at scale: a cat filed as a dog, a car filed as a truck.

You cannot check them all. Suppose you have budget for a human to look at 500 examples
out of 5,000. Which 500?

Every answer to that is a ranking. Rank the training set by some signal, hand the top
500 to the auditor, and the auditor corrects what is genuinely wrong and leaves the
rest alone. The budget is fixed and the remediation policy is fixed, so the only thing
that varies between methods is the order they put the examples in.

That is the question this project asks, and it has a second half that is easy to
forget. Finding more corrupted labels is not the goal. Having a better model is. Those
are not the same thing, and part of the work here is measuring how far apart they are.

So make the call now, on the half that matters: compared with auditing 500 examples at
random, how much better will a well-designed ranking make the final model? The results
table also reports what a perfect audit of the same 500 would have managed, so you can
see whether any shortfall is your ranking or the budget.

Answer it in the cell below before you read any further. One word, and it takes five seconds.

In [ ]:
# Your first call, before the notebook shows you anything. One word, then run the cell.
#
#   "none"    no difference you could measure
#   "small"   a real difference, but too small to act on
#   "large"   big enough to change what you would build
#
# Not graded, and deliberately made before you know the metric or the baseline. The
# results cell near the end compares this against what the run actually found. Being
# wrong here is the most useful outcome this project has to offer.

FIRST_CALL = ""

<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">The corruption is planted, so you know the answer key and can measure exactly what each method found. That is the only reason this is measurable at all. It is also the caveat: your ranking is tuned against one known noise structure, and real annotation error does not arrive with a manifest. Every lab in this course handed you the intervention. Here you author it yourself.</p>
</div>

## At a glance

| | |
|---|---|
| Runtime | roughly 4 to 8 minutes on a free Colab GPU runtime, much of it the one-time CIFAR-10 download |
| Data | CIFAR-10 with structured label noise planted in the training split |
| Model | a frozen ImageNet ResNet-18 with a linear head |
| You design | the ranking method, the hypothesis, and the smallest difference worth reporting |
| You submit | seven values the last cell prints, copied to the course page |

## What is graded

The last cell prints seven values and you copy them into the course page. Nothing asks
you to reproduce a number that moves from one run to the next.

| What it covers | When you get it |
|---|---|
| Three analysis helpers you write, run on inputs issued to you | after Task 3 |
| Your proposal: that the design holds, and what it will cost | before anything runs |
| The record: that the run met its contract, and what it spent | after the run |

The probes have definite answers. The helpers are the same whichever path you took, so
switching paths keeps that work, but the inputs are issued per learner and yours differ
from your classmates'. Two further questions ask you to read a result and decide what to
run next; each asks you to select every statement that follows, so there may be more than
one.

Your comparison will land in one of three places, and all three are complete
findings: the effect is at least as large as the one you declared worth acting on, it
is at most that large, or this many seeds cannot tell those apart. Reporting the third
honestly scores as well as reporting the first.

## Setup

The first cell fetches the shared contract layer. It runs in Colab and on a local
machine, and it does nothing if the files are already present.

In [ ]:
# Colab bootstrap: fetch the shared capstone modules if they are not already here.
import hashlib
import os
import urllib.request

REPO = ("https://raw.githubusercontent.com/codey-m/deep_learning/main/"
        "final_project/helpers")
NEEDED = ["project_schema.py", "label_debug_adapter.py"]
# Digests of the exact module versions this notebook was built and tested against. A
# file that does not match is a stale copy from an earlier session or a truncated
# download, and either one would fail later in a way that looks like your mistake.
DIGESTS = {
    "label_debug_adapter.py": "6737b36ebc55745d8f61743f9658f7f7d69020627a82834904a11a2234d46a8c",
    "project_schema.py": "41d249621c4b4a0ab151d3355c32763a729b37cf90e760549b4688a95a826bd7",
}


def digest_of(path):
    with open(path, "rb") as handle:
        return hashlib.sha256(handle.read()).hexdigest()


for name in NEEDED:
    if not os.path.exists(name) or digest_of(name) != DIGESTS[name]:
        urllib.request.urlretrieve(f"{REPO}/{name}", name)
    if digest_of(name) != DIGESTS[name]:
        raise RuntimeError(
            f"{name} does not match the version this notebook was tested against. "
            f"Delete it and restart the runtime.")
print("contract layer ready:", ", ".join(NEEDED))

In [ ]:
import math
import statistics
import time

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

import project_schema as schema
import label_debug_adapter as adapter
from label_debug_adapter import AuditProtocol
from project_schema import Contrast, ProjectPlan, RunRecord

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu")

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# QUICK_MODE is the graded default and is what the runtime estimate above assumes.
# Turning it off replicates over more seeds, which narrows every noise floor
# without changing the design. That is the remedy when a contrast you care about
# comes out inconclusive: more seeds rather than a softer claim.
QUICK_MODE = True
RESOLUTION = 160
TRAIN_PER_CLASS, EVAL_PER_CLASS = 500, 200
BUDGET = 500
CORRUPT_RATE = 0.30
# Structured, not uniform. Four source classes lose labels to a plausible neighbour,
# which is what annotation error actually looks like.
CORRUPT_PAIRS = ((3, 5), (1, 9), (4, 7), (2, 0))
CLASS_NAMES = ("plane", "auto", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck")
HEAD_EPOCHS = 80
SEEDS = tuple(7960 + i for i in range(5 if QUICK_MODE else 10))
METRIC = "recall_at_k"

print(f"device: {DEVICE}")
if DEVICE.type == "cpu":
    print("\nWARNING: no GPU is available, so this will run on CPU. The runtime above "
          "assumes a GPU and this will take several times longer. In Colab, use "
          "Runtime > Change runtime type > GPU, then run this cell again.")
print(f"planted corruption: " + ", ".join(
    f"{CLASS_NAMES[a]}->{CLASS_NAMES[b]}" for a, b in CORRUPT_PAIRS))
print(f"{CORRUPT_RATE:.0%} of each source class, audit budget {BUDGET}")

### Features, and why the backbone is frozen

A frozen ImageNet ResNet-18 turns each image into a 512-dimensional vector once, and
every model in this project is a linear head on top of those vectors. That keeps the
whole experiment inside a Colab session, and it means the only thing that changes
between conditions is the labels the head is trained on.

In [ ]:
train_set = datasets.CIFAR10("data", train=True, transform=transforms.ToTensor(),
                             download=True)
test_set = datasets.CIFAR10("data", train=False, transform=transforms.ToTensor(),
                            download=True)
train_targets = torch.as_tensor(train_set.targets)
test_targets = torch.as_tensor(test_set.targets)

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
trunk = nn.Sequential(*list(backbone.children())[:-1]).to(DEVICE).eval()
BACKBONE_PARAMETERS = sum(p.numel() for p in trunk.parameters())
FEATURE_DIM = 512
HEAD_PARAMETERS = FEATURE_DIM * 10 + 10


@torch.no_grad()
def embed(dataset, indices):
    """One pooled 512-d vector per image."""
    out = []
    for images, _ in DataLoader(Subset(dataset, indices), batch_size=128,
                                shuffle=False):
        images = nn.functional.interpolate(images, size=(RESOLUTION, RESOLUTION),
                                           mode="bilinear", align_corners=False)
        out.append(trunk(((images - MEAN) / STD).to(DEVICE)).flatten(1).cpu())
    return torch.cat(out)


def split(seed):
    """A fresh class-balanced train and evaluation split for each seed."""
    generator = torch.Generator().manual_seed(seed)
    train, evaluate = [], []
    for class_id in range(10):
        idx = torch.where(train_targets == class_id)[0]
        train += idx[torch.randperm(len(idx), generator=generator)][
            :TRAIN_PER_CLASS].tolist()
        pos = torch.where(test_targets == class_id)[0]
        evaluate += pos[torch.randperm(len(pos), generator=generator)][
            :EVAL_PER_CLASS].tolist()
    return sorted(train), sorted(evaluate)


def plant_corruption(labels, seed):
    """Flip a fixed fraction of specific class pairs. Returns noisy labels and mask."""
    generator = torch.Generator().manual_seed(seed + 5000)
    noisy = labels.clone()
    mask = torch.zeros(len(labels), dtype=torch.bool)
    for source, target in CORRUPT_PAIRS:
        positions = torch.where(labels == source)[0]
        chosen = positions[torch.randperm(len(positions), generator=generator)]
        chosen = chosen[:int(round(CORRUPT_RATE * len(positions)))]
        noisy[chosen] = target
        mask[chosen] = True
    return noisy, mask


print(f"frozen backbone parameters: {BACKBONE_PARAMETERS:,}")
print(f"trainable head parameters:  {HEAD_PARAMETERS:,}")

### The audit protocol

The protocol is fixed for every method, and that is what makes the comparison a
question about ranking rather than a question about who was allowed to do more.

Each method returns exactly `BUDGET` training indices. A simulated auditor inspects
those and only those. Where a label was genuinely corrupted, the auditor restores the
true one. Where it was already correct, the auditor changes nothing and the budget is
spent for no gain. Then a fresh head is trained on the repaired labels, and every
condition pays for exactly the same amount of training whether its ranking used the
cross-fit folds or not.

In [ ]:
protocol = AuditProtocol(BUDGET, CORRUPT_RATE, CORRUPT_PAIRS)
print(protocol.describe())

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Your ranking method never sees which labels were flipped. The contract inspects each method's arguments and rejects any that accepts something outside the permitted set, because a method that can read the answer key is not a method. Note that this is a documented protocol rather than a sandbox: Python cannot stop a closure, so it catches the accident and records the intent.</p>
</div>

### Before Task 1: your issued inputs

Checkpoint 1 on the course page issues you a parameter count, a step count, a batch
size, and a three-seed table of (baseline, treatment) pairs. They are drawn per learner,
so yours differ from your classmates' and the three values you submit have to come from
your own helpers.

Copy them into the block below. You can leave the zeros for now: every probe below runs
either way, and the fixed probes are what tell you your helpers work.

In [ ]:
# From Checkpoint 1 on the course page. Replace the zeros with your issued values.
ISSUED_PARAMETERS = 0
ISSUED_STEPS = 0
ISSUED_BATCH = 0
ISSUED_TABLE = {
    1: (0.0, 0.0),
    2: (0.0, 0.0),
    3: (0.0, 0.0),
}

issued_ready = ISSUED_PARAMETERS > 0 and ISSUED_TABLE[1] != (0.0, 0.0)
print("issued inputs:", "loaded" if issued_ready else "not filled in yet")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 1</div>
<div style="font-weight:700;font-size:1.2rem;">Scope a run to a compute budget</div>
</div>

Every lab so far handed you a compute budget. Estimating one is a skill in itself, and
it is the difference between an experiment that finishes inside a Colab session and
one that dies partway through with nothing to show.

Use the cheapest estimate that tracks real cost: the parameters that do work on
each example, multiplied by the number of examples pushed through. Here that is the linear heads, which
receive gradients, plus the frozen backbone's forward passes, which are counted
separately below because embedding every image dominates the total. Return the count in
billions, so the numbers stay readable:

$$\text{budget units} = \frac{\text{trainable parameters} \times \text{steps} \times \text{batch size}}{10^9}$$

<div style="border-left:6px solid #1D4ED8;background:#eff6ff;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#1D4ED8;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Intuition</div>
<p style="margin:6px 0 0;">This is a proxy, not a stopwatch. It will not predict wall-clock seconds on a particular GPU, and it is not meant to. What it does is rank designs against each other and catch the one that is a hundred times larger than you thought.</p>
</div>

In [ ]:
# STUDENT TASK 1: estimate the cost of a run in budget units.
def compute_budget(trainable_parameters, steps, batch_size):
    """Trainable parameters times examples processed, in billions."""
    # TODO: return trainable_parameters * steps * batch_size, expressed in billions.
    return 0.0

In [ ]:
# Probe 1 (fixed input, definite answer): 35,000 parameters, 250 steps, batch size 50.
# This one never changes, so it tells you whether compute_budget works at all.
probe_budget_value = round(compute_budget(35_000, 250, 50), 4)
budget_probe_contract = abs(probe_budget_value - 0.4375) < 1e-4
print(f"R1 probe: {probe_budget_value} budget units "
      f"({'matches' if budget_probe_contract else 'does not match'} the expected 0.4375)")

# The value you submit comes from your own issued inputs.
if issued_ready:
    issued_budget_value = round(
        compute_budget(ISSUED_PARAMETERS, ISSUED_STEPS, ISSUED_BATCH), 4)
    print(f"R1 to submit: {issued_budget_value} budget units")
else:
    issued_budget_value = None
    print("R1 to submit: fill in the issued block above first")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 2</div>
<div style="font-weight:700;font-size:1.2rem;">Compare two conditions the paired way</div>
</div>

You will run each method under five random seeds. A seed changes which images land in
the split, which labels get flipped, and how the head initializes, so the same method
does not give the same recall twice.

The naive comparison takes the mean of one condition and subtracts the mean of the
other. The paired comparison takes the difference within each seed first, then
averages those differences.

With the same seeds in both conditions these two give the identical mean. What changes
is the uncertainty around it. One seed might plant an unusually easy corruption for
every method at once, and when you difference within that seed, its easiness cancels.
Differencing the group means leaves it in.

How much pairing buys depends on how much the two conditions actually share, so it
is not a fixed number and is not worth taking on faith. When you read the result you
will see both floors, the paired one and the one you would have got by differencing
the group means, and you can judge the size of the difference on your own run.

In [ ]:
# STUDENT TASK 2: the mean within-seed difference between two conditions.
def paired_difference(treatment_by_seed, reference_by_seed):
    """Both arguments map seed -> metric value. Use only the seeds present in both."""
    seeds = sorted(set(treatment_by_seed) & set(reference_by_seed))
    # TODO: build the list of within-seed differences (treatment minus reference),
    # then return their mean.
    return 0.0

In [ ]:
# Probe 2 (fixed input, definite answer): three seeds, two conditions.
PROBE_TREATMENT = {1: 0.30, 2: 0.28, 3: 0.26}
PROBE_REFERENCE = {1: 0.34, 2: 0.33, 3: 0.29}
probe_paired_value = round(paired_difference(PROBE_TREATMENT, PROBE_REFERENCE), 4)
paired_probe_contract = abs(probe_paired_value - (-0.04)) < 1e-4
print(f"R2 probe: {probe_paired_value} "
      f"({'matches' if paired_probe_contract else 'does not match'} the expected -0.04)")

if issued_ready:
    issued_treatment = {seed: ISSUED_TABLE[seed][1] for seed in (1, 2, 3)}
    issued_reference = {seed: ISSUED_TABLE[seed][0] for seed in (1, 2, 3)}
    issued_paired_value = round(
        paired_difference(issued_treatment, issued_reference), 4)
    print(f"R2 to submit: {issued_paired_value}")
else:
    issued_paired_value = None
    print("R2 to submit: fill in the issued block above first")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 3</div>
<div style="font-weight:700;font-size:1.2rem;">Decide when a difference is big enough to believe</div>
</div>

A mean difference on its own settles nothing. Five seeds is a small sample, and a
difference of two recall points means one thing when the seed-to-seed spread is half a
point and another thing entirely when the spread is ten points.

The floor is the half-width of a t-interval on the paired differences:

$$\text{floor} = t_{0.975,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

where $s$ is the sample standard deviation of the within-seed differences and $n$ is
how many you have. A difference is resolved when its magnitude exceeds its own floor.
Anything smaller is inside the noise your own seeds produce, and you cannot tell it
apart from zero.

The critical value is supplied. You combine the pieces.

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Resolving a difference and caring about one are separate questions, and the verdict below combines them into a single call. If the whole interval sits beyond your SESOI, the effect is at least the size you said you would act on. If the whole interval sits inside it, the effect is at most that size, which is a real result and not a failure. If the interval straddles your SESOI, the honest answer is that this many seeds cannot tell, and the fix is more seeds rather than a softer claim.</p>
</div>

In [ ]:
# STUDENT TASK 3: the paired noise floor for a list of within-seed differences.
def resolution_floor(differences, critical_value):
    """Half-width of the t-interval around the mean of ``differences``."""
    count = len(differences)
    # TODO: return critical_value * (sample standard deviation) / sqrt(count).
    return 0.0

In [ ]:
# Probe 3 (fixed input, definite answer): the same three seeds as probe 2.
PROBE_DIFFERENCES = [PROBE_TREATMENT[s] - PROBE_REFERENCE[s] for s in (1, 2, 3)]
probe_floor_value = round(
    resolution_floor(PROBE_DIFFERENCES, schema._critical(len(PROBE_DIFFERENCES))), 4)
floor_probe_contract = abs(probe_floor_value - 0.0248) < 1e-4
print(f"R3 probe: {probe_floor_value} "
      f"({'matches' if floor_probe_contract else 'does not match'} the expected 0.0248)")
print(f"the probe difference of {probe_paired_value} is "
      f"{'resolved' if abs(probe_paired_value) > probe_floor_value else 'inside the noise'}")

if issued_ready:
    issued_differences = [ISSUED_TABLE[seed][1] - ISSUED_TABLE[seed][0]
                          for seed in (1, 2, 3)]
    issued_floor_value = round(
        resolution_floor(issued_differences,
                         schema._critical(len(issued_differences))), 4)
    print(f"R3 to submit: {issued_floor_value}")
    print(f"your issued difference of {issued_paired_value} is "
          f"{'resolved' if abs(issued_paired_value) > issued_floor_value else 'inside the noise'}")
else:
    issued_floor_value = None
    print("R3 to submit: fill in the issued block above first")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 4</div>
<div style="font-weight:700;font-size:1.2rem;">Design your ranking method</div>
</div>

This is the part that is yours. Write a function that returns exactly `budget`
training indices: the ones you would hand to the auditor.

You are given four signals and may use any of them.

`features` are the 512-dimensional vectors, so you can ask geometric questions like
whether an example sits among its own class.

`noisy_labels` are the labels as recorded, including the wrong ones.

`logits` come from a head trained on the whole training split with its noisy labels.
A model fit to its own noise will often fit the wrong labels too, which is exactly the
weakness this signal has.

`fold_logits` come from cross-fitting: the split is halved, a head is trained on one
half, and it scores the other. Each example is scored by a model that never saw it,
so a memorized wrong label cannot hide behind having been trained on.

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Return exactly the budget, with no duplicates. A method that returns 501 indices is not competing on the same terms, and a method that returns the same example twice has spent budget on nothing. Both are rejected.</p>
</div>

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Signals worth considering</summary>
<div style="margin-top:8px;"><ul><li><b>Loss on the observed label.</b> The supplied baseline. High cross-entropy against the recorded label means the model disagrees with it.</li><li><b>Cross-fit margin.</b> The observed label's logit minus the best competing logit, computed from the held-out folds. The most negative examples are those a model disagrees with when it never trained on them.</li><li><b>Neighbourhood disagreement.</b> Look at an example's nearest neighbours in feature space and ask what fraction carry a different label. This uses no trained model at all.</li><li><b>Combining signals.</b> Two rankings can be averaged, or one can break ties in the other. Combination is a design decision like any other, and it is worth asking whether it beats either signal alone.</li><li><b>Class awareness.</b> The corruption is structured, not uniform, and the affected class pairs are printed at the top of this notebook. A method may use that, and one that spends its budget where the noise actually is should find more than one that spreads it evenly. It is a legitimate design, but it is debugging a generator you were shown rather than noise you discovered, and your caveat has to say so.</li></ul></div>
</details>

In [ ]:
# STUDENT TASK 4: author your ranking method.
def learner_rank(features, noisy_labels, logits, fold_logits, budget):
    """Return exactly ``budget`` distinct positions to hand to the auditor.

    features:     (N, 512) frozen backbone vectors
    noisy_labels: (N,) labels as recorded, some of them wrong
    logits:       (N, 10) from a head trained on all of it, noise included
    fold_logits:  (N, 10) from cross-fitting, so each score is out of sample
    """
    # TODO: replace this with your own ranking. Returning the loss ranking would
    # reproduce the supplied baseline exactly, which the design contract rejects.
    loss = nn.functional.cross_entropy(logits, noisy_labels, reduction="none")
    return torch.argsort(loss, descending=True)[:budget].tolist()


MY_METHOD_REASON = ""  # TODO: one sentence, at least 6 words, on what your
                       # ranking sees that the loss baseline misses.

In [ ]:
def rank_random(noisy_labels, budget, rng):
    """The control: spend the budget without looking at anything."""
    return torch.randperm(len(noisy_labels), generator=rng)[:budget].tolist()


def rank_loss(logits, noisy_labels, budget):
    """Supplied baseline: highest cross-entropy on the observed label."""
    loss = nn.functional.cross_entropy(logits, noisy_labels, reduction="none")
    return torch.argsort(loss, descending=True)[:budget].tolist()


# Does the method behave like a method? Checked on a fixed probe, before anything
# expensive runs.
_gen = torch.Generator().manual_seed(3)
_n, _k = 400, 50
_probe = {
    "features": torch.randn(_n, 512, generator=_gen),
    "noisy_labels": torch.randint(0, 10, (_n,), generator=_gen),
    "logits": torch.randn(_n, 10, generator=_gen),
    "fold_logits": torch.randn(_n, 10, generator=_gen),
    "budget": _k,
}
_picked = learner_rank(**_probe)
method_is_well_formed = (
    len(_picked) == _k
    and len(set(_picked)) == _k
    and all(0 <= int(i) < _n for i in _picked))
# Compared as sets, and by how much they overlap. The auditor inspects a set of
# examples, so handing back the loss baseline's selections in a different order is the
# same audit, and an order-sensitive comparison would have called it a new method.
_loss_picks = set(rank_loss(_probe["logits"], _probe["noisy_labels"], _k))
overlap = len(set(_picked) & _loss_picks) if method_is_well_formed else 0
method_is_novel = method_is_well_formed and overlap < 0.9 * _k

print(f"well formed: {method_is_well_formed}   differs from the loss baseline: "
      f"{method_is_novel}")
if method_is_well_formed:
    print(f"on the probe your ranking shares {overlap} of {_k} picks with the loss "
          f"baseline; more than {int(0.9 * _k)} would be the same audit")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 5</div>
<div style="font-weight:700;font-size:1.2rem;">Declare your prediction before you measure</div>
</div>

Four declarations, all made before a single head is trained.

**The hypothesis.** What you expect your ranking to do, and the mechanism you think
would cause it.

**The required contrast.** Which comparison your project stands on. Your method
against random asks whether ranking helps at all. Your method against loss asks the
harder question, whether your signal beats the obvious one.

**The smallest effect worth caring about, twice.** This project asks two questions and
so needs two thresholds. In recall points, declaring 0.05 says finding five percent
more of the corruption would change which method you would run. In accuracy points,
declaring 0.05 says a five-point gain in worst-class accuracy would change whether you
would commission the audit at all. They are different questions and they do not have
to take the same number.

**The budget.** What the whole experiment will cost, from Task 1.

<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">Declaring the effect size you care about before measuring is what separates an experiment from a search. If you fix it afterwards, you will fix it wherever your result happened to land, and you will have learned nothing you did not already assume.</p>
</div>

In [ ]:
# STUDENT TASK 5: declare the design.
MY_HYPOTHESIS = ""  # TODO: at least 12 words. What you expect, and the
                    # mechanism you think causes it.

# TODO: which comparison does your project stand on? treatment is always
# "learner_method"; reference is "random" or "loss"; direction is "greater" (your
# ranking finds more) or "less" (it finds less).
REQUIRED_CONTRAST = Contrast("learner_vs_random", "learner_method", "random",
                             required=True, direction="greater")

# TODO: the smallest change in recall at k you would act on.
# Must be above 0 and at most 0.5.
SESOI_ABSOLUTE = 0.05

# TODO: the smallest change in worst-class accuracy you would act on, above 0
# and at most 0.5. Without this the
# downstream half of the question has no declared threshold, and any reading of it
# afterwards would be exploratory rather than predicted.
SESOI_DOWNSTREAM = 0.05

In [ ]:
CONDITIONS = ("random", "loss", "learner_method")
METHODS = {"random": rank_random, "loss": rank_loss, "learner_method": learner_rank}
CONTRASTS = (
    REQUIRED_CONTRAST,
    Contrast("loss_vs_random", "loss", "random", direction="greater"),
    Contrast("learner_vs_loss", "learner_method", "loss", direction="greater"),
)
configs = {c: {"ranking_method": c, "budget": BUDGET, "corrects_found": True}
           for c in CONDITIONS}

images_per_seed = (TRAIN_PER_CLASS + EVAL_PER_CLASS) * 10
# One base head, two cross-fit halves (one full-split equivalent between them), one
# repaired head per condition, and the two oracle bookends. Every condition pays for all
# of it, whether its ranking used the folds or not, so the comparison is about ranking
# rather than about compute.
heads_per_seed = 1 + 1 + len(CONDITIONS) + 2
projected_budget_units = round(
    BACKBONE_PARAMETERS * images_per_seed * len(SEEDS) / 1e9
    + compute_budget(HEAD_PARAMETERS, HEAD_EPOCHS, TRAIN_PER_CLASS * 10)
    * heads_per_seed * len(SEEDS), 4)

plan = ProjectPlan(
    path="label_debugging",
    question="Under a fixed audit budget, which ranking finds the most corrupted "
             "training labels, and what does correcting them buy downstream?",
    hypothesis=MY_HYPOTHESIS,
    control="random",
    intervention="ranking_method",
    declared_change="ranking_method",
    conditions=CONDITIONS,
    evaluation_slices=(adapter.TRAIN_SLICE, adapter.EVAL_SLICE),
    seeds=SEEDS,
    compute_budget=projected_budget_units,
    decisions=(f"budget={BUDGET}", f"rate={CORRUPT_RATE}",
               f"sesoi={SESOI_ABSOLUTE}", f"sesoi_downstream={SESOI_DOWNSTREAM}"),
    condition_kind="categorical",
    contrasts=CONTRASTS,
)

BUDGET_MIN, BUDGET_MAX = 0.0, 20000.0

# The generic proposal contract, run now rather than after the experiment. This is the
# same check the execution contract applies to the plan, so a malformed design fails here
# in a second instead of after the run.
proposal = schema.ContractResult()
schema.check_plan(plan, budget_min=BUDGET_MIN, budget_max=BUDGET_MAX, result=proposal)

design_contract, design_report = schema.checklist({
    "your method returns exactly the budget, with no duplicates": method_is_well_formed,
    "your ranking differs from the supplied loss baseline": method_is_novel,
    "MY_HYPOTHESIS is at least 12 words": len(MY_HYPOTHESIS.split()) >= 12,
    "MY_METHOD_REASON is at least 6 words": len(MY_METHOD_REASON.split()) >= 6,
    "your required contrast is marked required": REQUIRED_CONTRAST.required,
    "its treatment is 'learner_method'":
        REQUIRED_CONTRAST.treatment == "learner_method",
    "its reference is 'random' or 'loss'":
        REQUIRED_CONTRAST.reference in ("random", "loss"),
    "its direction is 'less' or 'greater'":
        REQUIRED_CONTRAST.direction in ("less", "greater"),
    "SESOI_ABSOLUTE is above 0 and at most 0.5": 0.0 < SESOI_ABSOLUTE <= 0.5,
    "SESOI_DOWNSTREAM is above 0 and at most 0.5": 0.0 < SESOI_DOWNSTREAM <= 0.5,
    "your projected budget is above zero (Task 1 is finished)":
        projected_budget_units > 0,
    "the plan passes the generic proposal checks printed below": bool(proposal.passed),
})

print(f"R4 design contract: {design_contract}")
print(design_report)
if not proposal.passed:
    print(proposal.report())
print(f"R5 projected budget: {projected_budget_units} units for "
      f"{len(CONDITIONS)} methods x {len(SEEDS)} seeds")
print(f"plan digest: {plan.freeze()}")

# Bind every measurement to the design and to the code that produced it. The bytecode
# and constants of your own function go into the digest alongside the frozen plan, so a
# result table can be traced to the exact method that made it and a plan edited after
# the run stops matching its own rows. A notebook can always be re-run from the top,
# so this detects a changed design rather than preventing one.
provenance = hashlib.sha256(
    plan.freeze().encode("utf-8")
    + learner_rank.__code__.co_code
    + repr(learner_rank.__code__.co_consts).encode("utf-8")).hexdigest()[:16]
print(f"run provenance: {provenance}")

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">R4 must read 1 before you go on. If it does not, one of the declarations above is still a placeholder, or your method still returns the loss ranking. Running the experiment on a design that fails its own contract wastes the compute and the result will not be gradeable.</p>
</div>

## Run the experiment

Per seed: draw a split, plant the corruption, train the base head and the two
cross-fit halves, then run each method, audit its picks, and retrain on the repaired
labels.

In [ ]:
def fit_head(features, labels, seed):
    """A linear head on frozen features. Full-batch, so the step count is exact."""
    torch.manual_seed(seed)
    head = nn.Linear(features.shape[1], 10).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-4)
    inputs, targets = features.to(DEVICE), labels.to(DEVICE)
    for _ in range(HEAD_EPOCHS):
        optimizer.zero_grad()
        nn.functional.cross_entropy(head(inputs), targets).backward()
        optimizer.step()
    return head, compute_budget(HEAD_PARAMETERS, HEAD_EPOCHS, len(inputs))


@torch.no_grad()
def logits_of(head, features):
    return head(features.to(DEVICE)).cpu()


def macro_and_worst(prediction, truth):
    per_class = []
    for class_id in range(10):
        picked = truth == class_id
        if picked.any():
            per_class.append((prediction[picked] == truth[picked]).float().mean().item())
    return statistics.mean(per_class), min(per_class)


started = time.time()
records, selections, train_ids_by_seed = [], {}, {}
found_counts = {}
bookends = {"no_repair": [], "best_at_budget": [], "oracle": []}
method_seconds = {}
pristine_eval, observed_eval = [], []
measured_budget_units = 0.0

for seed in SEEDS:
    train_idx, eval_idx = split(seed)
    train_ids_by_seed[seed] = train_idx
    split_id = schema.split_hash(train_idx, eval_idx)

    train_z = embed(train_set, train_idx)
    eval_z = embed(test_set, eval_idx)
    train_y = train_targets[train_idx]
    eval_y = test_targets[eval_idx]
    # The corruption is planted in training only. Evaluation labels stay pristine, and
    # the contract checks it, because downstream accuracy against moving labels is
    # meaningless.
    measured_budget_units += BACKBONE_PARAMETERS * (len(train_idx)
                                                    + len(eval_idx)) / 1e9
    # Two independent sources, accumulated over every seed: the dataset's own
    # targets, and the labels actually handed to the scorer below. An earlier version
    # compared one copy of eval_y against another copy of itself, for the first seed
    # only, so it could not have failed whatever the experiment did to its labels.
    pristine_eval += [int(test_set.targets[i]) for i in eval_idx]
    observed_eval += [int(v) for v in eval_y]

    noisy_y, mask = plant_corruption(train_y, seed)

    base_head, spent = fit_head(train_z, noisy_y, seed)
    measured_budget_units += spent
    base_logits = logits_of(base_head, train_z)
    # Bookend 1, no repair: the model you already have, trained on the labels exactly
    # as recorded. Random auditing is not this baseline, because a random audit still
    # corrects whatever corruption it happens to land on.
    no_repair = macro_and_worst(logits_of(base_head, eval_z).argmax(1), eval_y)

    half = torch.zeros(len(train_idx), dtype=torch.bool)
    half[torch.randperm(len(train_idx),
                        generator=torch.Generator().manual_seed(seed)
                        )[:len(train_idx) // 2]] = True
    fold_logits = torch.zeros(len(train_idx), 10)
    for held in (True, False):
        trained, spent = fit_head(train_z[~(half == held)], noisy_y[~(half == held)],
                                  seed)
        measured_budget_units += spent
        fold_logits[half == held] = logits_of(trained, train_z[half == held])

    # Bookend 2, the ceiling: every corrupted label fixed, which no finite audit budget
    # could reach. Without it, beating random reads as success when it may have closed
    # very little of the gap that was actually available.
    oracle_head, spent = fit_head(train_z, train_y, seed)
    measured_budget_units += spent
    oracle = macro_and_worst(logits_of(oracle_head, eval_z).argmax(1), eval_y)

    # Bookend 3, the best any ranking could do at THIS budget: audit the corrupted
    # examples first and stop at BUDGET. It reads the answer key, so it is not a method
    # and cannot compete. It is the line that separates "my ranking is weak" from "the
    # budget is too small", which are different problems with different fixes.
    perfect = torch.where(mask)[0][:BUDGET]
    corrected_perfect = noisy_y.clone()
    corrected_perfect[perfect] = train_y[perfect]
    budget_head, spent = fit_head(train_z, corrected_perfect, seed)
    measured_budget_units += spent
    best_at_budget = macro_and_worst(
        logits_of(budget_head, eval_z).argmax(1), eval_y)

    bookends["no_repair"].append(no_repair)
    bookends["best_at_budget"].append(best_at_budget)
    bookends["oracle"].append(oracle)

    # Base head, both cross-fit halves, the two bookends, and the repaired head that
    # every condition trains. Equal for every condition, which is the point.
    effort = HEAD_EPOCHS * 6

    for condition in CONDITIONS:
        # R5 and R7 price the supplied backbone and heads. They cannot price a ranking
        # method, which may do anything from a sort to a nearest-neighbour search, so
        # the method is timed instead. That time is diagnostic and never graded: a
        # wall clock reading is not comparable between one machine and another.
        ranking_started = time.perf_counter()
        if condition == "random":
            picked = rank_random(noisy_y, BUDGET,
                                 torch.Generator().manual_seed(seed + 99))
        elif condition == "loss":
            picked = rank_loss(base_logits, noisy_y, BUDGET)
        else:
            picked = learner_rank(train_z, noisy_y, base_logits, fold_logits, BUDGET)
        ranking_seconds = time.perf_counter() - ranking_started
        method_seconds[condition] = method_seconds.get(condition, 0.0) + ranking_seconds
        selections[(condition, seed)] = [train_idx[i] for i in picked]

        chosen = torch.tensor(picked)
        found = int(mask[chosen].sum())
        found_counts[(condition, seed)] = found
        corrected = noisy_y.clone()
        corrected[chosen[mask[chosen]]] = train_y[chosen[mask[chosen]]]

        repaired, spent = fit_head(train_z, corrected, seed)
        measured_budget_units += spent
        prediction = logits_of(repaired, eval_z).argmax(1)
        macro, worst = macro_and_worst(prediction, eval_y)

        for name, value, slice_name in (
                ("precision_at_k", found / BUDGET, adapter.TRAIN_SLICE),
                ("recall_at_k", found / max(1, int(mask.sum())), adapter.TRAIN_SLICE),
                ("clean_audited", (BUDGET - found) / BUDGET, adapter.TRAIN_SLICE),
                ("macro_accuracy", macro, adapter.EVAL_SLICE),
                ("worst_class_accuracy", worst, adapter.EVAL_SLICE)):
            records.append(RunRecord(
                condition=condition, seed=seed, split_hash=split_id,
                config_hash=schema.config_hash(configs[condition]),
                training_steps=effort, runtime_seconds=ranking_seconds, provenance=provenance,
                metric_name=name, evaluation_slice=slice_name, value=value))
    print(f"  seed {seed} done ({time.time() - started:.0f}s elapsed)", flush=True)

measured_budget_units = round(measured_budget_units, 4)
print(f"\n{len(records)} records in {time.time() - started:.0f}s")
print(f"budget projected {projected_budget_units}, measured {measured_budget_units}")
print(f"\nthose cover the supplied backbone and heads only. Your ranking method itself "
      f"took {method_seconds['learner_method'] * 1000:.0f} ms across {len(SEEDS)} "
      f"seeds, against {method_seconds['loss'] * 1000:.0f} ms for the loss baseline. "
      f"That is a wall clock reading on this machine, so it is reported and not "
      f"graded; a method far slower than the baselines is worth mentioning in your "
      f"caveat even though nothing here checks it.")

<div style="border-left:6px solid #1D4ED8;background:#eff6ff;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#1D4ED8;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Intuition</div>
<p style="margin:6px 0 0;">R5 and R7 should agree closely here, because every condition's compute is fixed by the protocol before the run starts. That is not a check that cannot fail: it accumulates from each head actually fitted, so a design that trains the folds only for the method that uses them, or skips a repaired head, diverges immediately.</p>
</div>

## Read the result

Two tables, and the gap between them is the point of this project.

The first is about the audit: what fraction of the planted corruption each method
found, and how much of the budget it spent on labels that were already correct.

The second is about the model that came out the other end. That is what you actually
wanted.

In [ ]:
def mean_of(condition, name):
    return statistics.mean(r.value for r in records
                           if r.condition == condition and r.metric_name == name)


print("what the audit found")
print(f"{'method':<16}" + "".join(f"{h:>16}" for h in
                                  ("precision@k", "recall@k", "wasted budget")))
for condition in CONDITIONS:
    print(f"{condition:<16}" + "".join(
        f"{mean_of(condition, name):>16.3f}" for name in
        ("precision_at_k", "recall_at_k", "clean_audited")))

def bookend(name):
    macro = statistics.mean(v[0] for v in bookends[name])
    worst = statistics.mean(v[1] for v in bookends[name])
    return macro, worst


print("\nwhat the model did afterwards")
print(f"{'method':<22}" + "".join(f"{h:>16}" for h in
                                  ("macro accuracy", "worst class")))
floor_macro, floor_worst = bookend("no_repair")
print(f"{'no repair at all':<22}{floor_macro:>16.3f}{floor_worst:>16.3f}")
for condition in CONDITIONS:
    print(f"{condition:<22}" + "".join(
        f"{mean_of(condition, name):>16.3f}" for name in
        ("macro_accuracy", "worst_class_accuracy")))
budget_macro, budget_worst = bookend("best_at_budget")
print(f"{'a perfect audit of ' + str(BUDGET):<22}{budget_macro:>16.3f}"
      f"{budget_worst:>16.3f}")
ceiling_macro, ceiling_worst = bookend("oracle")
print(f"{'every label fixed':<22}{ceiling_macro:>16.3f}{ceiling_worst:>16.3f}")

# How much of the gain that was actually available did each method capture? Beating
# random says nothing about this, and it is the number a team would be asked for.
available = ceiling_worst - floor_worst
print(f"\nthe whole gap between no repair and every label fixed is "
      f"{available:+.3f} worst-class accuracy, of which a perfect audit of {BUDGET} "
      f"could reach {budget_worst - floor_worst:+.3f}")
for condition in CONDITIONS:
    captured = mean_of(condition, "worst_class_accuracy") - floor_worst
    share = captured / available if abs(available) > 1e-9 else float("nan")
    print(f"  {condition:<22} captured {captured:+.3f} of it ({share:.0%})")

print()
# Two questions, two declared thresholds, two verdicts. The audit metric is judged
# against your recall SESOI and the downstream metric against your accuracy SESOI,
# because "worth acting on" means a different number in each. Comparing the two
# verdicts is the whole point of this project.
report = schema.contrast_report(records, METRIC, adapter.TRAIN_SLICE, CONTRASTS,
                                sesoi=SESOI_ABSOLUTE)
down_report = schema.contrast_report(records, "worst_class_accuracy",
                                     adapter.EVAL_SLICE, CONTRASTS,
                                     sesoi=SESOI_DOWNSTREAM)
print(schema.format_contrasts(report, label=f"{METRIC} (what the audit found)"))
print()
print(schema.format_contrasts(down_report,
                              label="worst_class_accuracy (what the model did)"))

# Your own helpers, applied to your own experiment. They must agree with the report.
values = schema.paired_values(records, METRIC, adapter.TRAIN_SLICE)
required_row = next(row for row in report["rows"]
                    if row["name"] == REQUIRED_CONTRAST.name)
own_differences = [values[REQUIRED_CONTRAST.treatment][s]
                   - values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
own_mean = paired_difference(values[REQUIRED_CONTRAST.treatment],
                             values[REQUIRED_CONTRAST.reference])
own_floor = resolution_floor(own_differences, schema._critical(len(own_differences)))
helpers_agree = (abs(own_mean - required_row["mean"]) < 1e-9
                 and abs(own_floor - required_row["floor"]) < 1e-9)
print(f"\nyour helpers reproduce the report: {helpers_agree}")

# What pairing bought on this run: the same contrast judged by differencing the group
# means instead of within each seed. The factor depends on how much the two conditions
# share, so it is measured here rather than asserted.
_treat = [values[REQUIRED_CONTRAST.treatment][s] for s in SEEDS]
_ref = [values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
unpaired_floor = schema._critical(len(SEEDS)) * math.sqrt(
    (statistics.stdev(_treat) ** 2 + statistics.stdev(_ref) ** 2) / len(SEEDS))
if own_floor > 0:
    print(f"floor on your required contrast: {unpaired_floor:.4f} unpaired against "
          f"{own_floor:.4f} paired ({unpaired_floor / own_floor:.1f}x)")
    print("  pairing only buys something where the two conditions rise and fall "
          "together across seeds. A factor near 1.0 means yours barely do, which is a "
          "fact about the conditions rather than a mistake, and is worth a sentence in "
          "your caveat.")
else:
    print("floor comparison skipped: your resolution_floor returned 0, so Task 3 is "
          "not finished. Everything below this point depends on it.")

# The call you made at the top, before the notebook had shown you anything, set against
# what the run found. The opening question was about the model, so it is judged
# against the downstream metric rather than against the ranking metric the
# required contrast uses. It still follows your required contrast, which compares
# against random unless you changed the reference in Task 5. If you did, read
# this line against the contrast you chose.
first_call = str(globals().get("FIRST_CALL", "")).strip().lower()
if first_call in ("none", "small", "large"):
    called_large = first_call == "large"
    verdict = next(row for row in down_report["rows"]
                  if row["name"] == REQUIRED_CONTRAST.name)["verdict"]
    if verdict == "inconclusive":
        outcome = ("this run cannot separate those two possibilities, so your call "
                   "stands untested rather than wrong")
    elif (verdict == "meaningful") == called_large:
        outcome = "the run agrees with you"
    else:
        outcome = ("the run disagrees with you, which is the more interesting of the "
                   "two outcomes and belongs in your record")
    print(f"\nyour first call was {first_call!r} and the effect came out {verdict}: "
          f"{outcome}.")
else:
    print("\nno first call was recorded at the top of the notebook")

In [ ]:
# Signed paired differences with their intervals, which is what the experiment
# actually measured. Plotting the magnitude as a multiple of the floor, as an earlier
# version did, drew a large bar for a result that pointed the wrong way.
figure, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
names = [row["name"] for row in report["rows"]]
positions = list(range(len(names)))

for axis, panel, title, unit, band in (
        (axes[0], report, "what the audit found", "recall at k", SESOI_ABSOLUTE),
        (axes[1], down_report, "what the model did afterwards",
         "worst-class accuracy", None)):
    means_ = [row["mean"] for row in panel["rows"]]
    errors = [row["floor"] for row in panel["rows"]]
    axis.errorbar(means_, positions, xerr=errors, fmt="o", color="#A31F34",
                  capsize=4, lw=2)
    axis.axvline(0, color="#111827", lw=1)
    if band is not None:
        axis.axvspan(-band, band, color="#1D4ED8", alpha=0.12)
        axis.text(0, len(names) - 0.4, "inside here is below your SESOI",
                  ha="center", fontsize=7, color="#1D4ED8")
    axis.set_yticks(positions)
    axis.set_yticklabels(names, fontsize=8)
    axis.set_xlabel(f"paired difference in {unit} (95% CI)")
    axis.set_title(title)

plt.tight_layout()
plt.show()

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 6</div>
<div style="font-weight:700;font-size:1.2rem;">Write the record</div>
</div>

Six fields. The last two are the ones that decide whether this is science.

`claim` and `evidence` say what you found and the numbers behind it. `caveat` says
what your design could not control. `not_supported` says what a reader might
reasonably conclude from your result that your result does not actually show.
`next_experiment` names the single change you would make next.

Three of these are checked for form, not for quality, and the checks exist because
each names a way the record gets filled in without being thought about. `caveat` has
to name something the design held constant, because that is the boundary of what your
result covers. `resolved_comparisons` has to use the verdicts the analysis actually
returned, so it is written from the output rather than from memory. And
`not_supported` has to say something your claim does not, because pasting the claim
back is not a limitation.

Before you write anything, look at the line the results cell printed comparing your
`FIRST_CALL` against what the run found. If the run disagreed with you, say so plainly:
`claim` reports what happened, and `not_supported` is not the place to quietly rewrite
what you expected. A capstone that changed your mind is worth more than one that
confirmed it.

Look at what the audit found, at what the model did afterwards, and at the share of
the available gain each method captured. If a method found substantially more
corruption but the downstream difference did not clear its floor, then what you have
shown is that the ranking is better and not that the model is, and `claim` has to say
the first while `not_supported` says the second. That distinction is most of the grade
on this task, and it is the one that transfers directly to work: a proxy metric that
moves is not a result until the thing you actually cared about moves too.

In [ ]:
# STUDENT TASK 6: the experiment record. Every field must be non-empty.
learner_record = {
    "claim": "",             # TODO: what you found, in one sentence, with the numbers.
    "evidence": "",          # TODO: the measurements that support the claim.
    "resolved_comparisons": "",  # TODO: which contrasts cleared their floor, and by how much.
    "caveat": "",            # TODO: what your design could not control.
    "not_supported": "",     # TODO: what your result does NOT show.
    "next_experiment": "",   # TODO: the single change you would make next.
}

## The contract over what you ran

The checks below are the ones a research group would run before believing its own
result. Four are specific to this project.

Every method must spend exactly its budget, with no duplicates and nothing outside its
own training split. The corruption must stay confined to training, so the evaluation
labels are compared against the pristine ones. And no method may accept the corruption
mask as an argument, because a method that can read the answer key is not a method.

In [ ]:
result = adapter.run_all_checks(
    plan=plan, records=records, configs=configs, methods=METHODS,
    selections=selections, protocol=protocol,
    # Derived from the splits themselves, never from the selections under test: an
    # allow-list built from the selections would let an out-of-split pick authorise
    # itself.
    train_ids_by_seed=train_ids_by_seed, bookends=bookends,
    pristine_eval=pristine_eval, observed_eval=observed_eval,
    record=learner_record, budget_min=BUDGET_MIN, budget_max=BUDGET_MAX,
    metric=METRIC, slice_name=adapter.TRAIN_SLICE,
    # The downstream metrics are what this project calls the real objective, so they
    # are checked for completeness too rather than only the ranking metric.
    metrics=("precision_at_k", "clean_audited", "macro_accuracy",
             "worst_class_accuracy"),
    provenance=provenance, measured_budget=measured_budget_units)

execution_contract, execution_report = schema.checklist({
    "every contract check passed, listed above": bool(result.passed),
    "your own Task 2 and Task 3 helpers reproduce the reported analysis":
        helpers_agree,
    "your projected budget is above zero, so the comparison below means "
    "something": projected_budget_units > 0,
    "the compute you spent matches what you projected, within 10%":
        projected_budget_units > 0
        and abs(measured_budget_units - projected_budget_units)
        <= 0.10 * projected_budget_units,
})

print(f"contract passed: {result.passed}")
print(result.report())
print(f"\nR6 execution contract: {execution_contract}")
print(execution_report)

## Report values

Run the cell below once every task is complete and the experiment has finished. It
prints seven labelled values, R1 through R7. Copy each into the box with the matching
label on the course page.

R1 to R3 are your three analysis helpers run on the inputs Checkpoint 1 issued you, so
they are yours rather than the cohort's. R4 and R5 are the proposal you had before the
run. R6 and R7 are what the run actually did. The last two checkpoints ask how you read
your result and what you would run next, and are answered on the course page rather than
in the notebook.

In [ ]:
probe_budget = probe_budget_value if budget_probe_contract else -1.0
probe_paired = probe_paired_value if paired_probe_contract else -1.0
probe_floor = probe_floor_value if floor_probe_contract else -1.0

report_values = {
    "R1: compute budget on your issued inputs": issued_budget_value,
    "R2: paired difference on your issued table": issued_paired_value,
    "R3: resolution floor on your issued table": issued_floor_value,
    "R4: design contract": design_contract,
    "R5: projected budget units for your design": projected_budget_units,
    "R6: execution contract": execution_contract,
    "R7: measured budget units your experiment spent": measured_budget_units,
}
# A bare assertion at the end of a full run tells you something is wrong and nothing
# about what. Each line below names the task that produces it, so a failure points
# somewhere you can act.
ready, ready_report = schema.checklist({
    "issued inputs copied from Checkpoint 1 (block above Task 1)": issued_ready,
    "R1 matches its fixed probe (Task 1, compute_budget)":
        abs(probe_budget - 0.4375) < 1e-4,
    "R2 matches its fixed probe (Task 2, paired_difference)":
        abs(probe_paired - (-0.04)) < 1e-4,
    "R3 matches its fixed probe (Task 3, resolution_floor)":
        abs(probe_floor - 0.0248) < 1e-4,
    "R4 design contract passed (Tasks 4 and 5)": design_contract == 1,
    "R6 execution contract passed (the run and Task 6)": execution_contract == 1,
})
if not ready:
    print("Not ready to submit. Nothing is printed below until these pass:")
    print(ready_report)
else:
    print("CAPSTONE REPORT VALUES")
    for label, value in report_values.items():
        print(f"{label}: {value}")